# Capstone — Does a learned ranking beat a hand-written rule at choosing which pages a reviewer opens first?

**Lane 4 — CTR / Engagement Opportunity Scoring.** This notebook is the machine behind the deployed paper: every number, table and figure the page shows is produced here, in one top-to-bottom run, from the raw CSV. Nothing on the paper is typed by hand.

It compresses eight weeks — Week 1's question, Week 3's data contract, Week 4's transparent rule, Week 5's model, Week 6's validation audit, Week 7's action playbook — into the seven sections the paper mirrors.

> Working with an AI assistant? Tell it to read `skills/README.md` first, then load `writing-research-papers` + `deploying-static-pages`.

**Careful words throughout:** *observed / measured / directional / decision-support*. No causal claim appears in this notebook or on the paper, because no design here can carry one.

**Outputs:** figures to `docs/img/` (what the deployed page embeds, relative paths), metrics to `work/outputs/w08_capstone_metrics.json` (committed receipts), page to `docs/index.html`.

In [1]:
# ---- Setup ---------------------------------------------------------------------------------
import json, sys
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from IPython.display import display

SEED = 42
np.random.seed(SEED)

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / CSV).exists()), None)
if root is None:
    root = Path.cwd()
    df = pd.read_csv("https://raw.githubusercontent.com/nothaziq/FlyRank-ML-Week1/main/" + CSV)
else:
    df = pd.read_csv(root / CSV)

OUT = root / "work" / "outputs"
IMG = root / "docs" / "img"
OUT.mkdir(parents=True, exist_ok=True)
IMG.mkdir(parents=True, exist_ok=True)

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | scikit-learn {sklearn.__version__} | seed {SEED}")
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(f"figures -> {IMG.relative_to(root)}/   metrics -> {OUT.relative_to(root)}/")

python 3.12.10 | pandas 3.0.6 | scikit-learn 1.9.1 | seed 42
30,000 rows x 44 columns
figures -> docs\img/   metrics -> work\outputs/


## 1. Question

*The research question and the decision it supports.*

**The decision.** A FlyRank content reviewer has roughly 50 review slots in a week and a portfolio of tens of thousands of pages. Before this work, the order those pages got opened in was set by client priority and by whoever asked most recently. That is not a bad process — it is just an unmeasured one.

**The research question, as posed in Week 1 and unchanged since:**

> *Among pages that are already visible in search, can a learned ranking identify — better than a transparent hand-written rule — the ones whose click shortfall is still there a month later?*

Three things in that sentence are doing work:

- **"already visible"** — the question is about pages ranking 1–20 with enough impressions to measure. Pages nobody sees are a different problem (a ranking problem, not a click-through one).
- **"click shortfall"** — a page earning fewer clicks than pages at its position band typically do. Position is the dominant driver of CTR, so comparing a page to *its own band* is what makes the comparison fair.
- **"still there a month later"** — the part that makes this measurable at all. A one-month dip is noise; a gap that persists into a second window is the thing worth a human's time.

**What the answer is for.** Ordering a review queue. Not forecasting, not valuing, not deciding what to write. The strongest honest form of the output is *"open these first, and here is why."*

In [2]:
# ---- The question, in one number: how much of the portfolio is even in scope? ----------------
BANDS = [0, 3, 5, 7, 10, 15, 20]
BAND_LABELS = ["0-3", "3-5", "5-7", "7-10", "10-15", "15-20"]
IMP_FLOOR_30 = 500 / 3          # Week-4's 500-impression floor on a 90d window, put on a 30d window
MIN_EXP_CLICKS_30 = 10 / 3      # Week-4's 10 expected clicks, likewise
SHORTFALL = 0.5                 # "under half the band norm"

d = df.copy()
d["band"] = pd.cut(d.avg_position, BANDS, labels=BAND_LABELS)
visible = d.band.notna()
print(f"Pages in the slice                      : {len(d):,}")
print(f"  visible in search (position 1-20)     : {int(visible.sum()):,} ({visible.mean():.0%})")
print(f"  ...and measurable in the earlier window: "
      f"{int((visible & (d.impressions_prev_30d >= IMP_FLOOR_30)).sum()):,}")
print("\nEverything in this paper is about that last group. The rest is out of scope by construction,")
print("not by preference -- a CTR question needs impressions to be answerable.")

Pages in the slice                      : 30,000
  visible in search (position 1-20)     : 20,256 (68%)
  ...and measurable in the earlier window: 11,519

Everything in this paper is about that last group. The rest is out of scope by construction,
not by preference -- a CTR question needs impressions to be answerable.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used: the anonymized starter slice — `data/raw/content_refresh_anonymized.csv`, 30,000 rows × 44 columns.** Not the full ~79M-row warehouse release. That release exists and is documented in `docs/data-dictionary.md`, but this work did not use it, and saying otherwise would be the first dishonest sentence in the paper. Everything below describes the 30k slice.

**One table.** A page-level export: one row per content item, per client. Each row carries static metadata (word count, content type, keyword intent, age, freshness), 90-day totals, and two consecutive 30-day windows of Google Search Console and GA4 measurements.

**Date windows.** The slice carries **no calendar dates** — the windows are relative to export time: a trailing 90-day window, and inside it two consecutive 30-day windows (`*_prev_30d`, `*_last_30d`). That has a consequence the paper states plainly: **seasonality cannot be examined, and no result here is anchored to a period.** It also drives the whole design — the earlier window is treated as "what is knowable at decision time" and the later one as "the outcome".

**Anonymization, and what that removes.** Clients, content items and queries arrive as opaque hashes; there are no client names, domains, URLs, page titles or raw search queries anywhere in the file. So this work can say *what measurable pattern a page had*, never *what the page was about*. That limitation shows up again in the recommendations: the queue is structurally blind to topic.

**What was excluded, and why:**

| Excluded | Rows | Why |
|---|---|---|
| Pages with no position data (`avg_position = 0`) or position > 20 | — | band norms only exist for visible pages; `0` means "no data", not "rank zero" |
| Pages under ~167 impressions in the earlier 30-day window | — | a CTR computed on a handful of impressions is noise, not a measurement |
| Pages under ~167 impressions in the *later* window | 1,683 | the outcome would not be measurable — this is a **survivorship exclusion** and is reported as one |

The third exclusion is the uncomfortable one and it is named everywhere it matters: pages whose traffic collapsed between windows are removed from the evaluation, so every rate in this paper describes pages that stayed measurable.

In [3]:
# ---- Build the two windows, the universe, and the exclusion ledger ---------------------------
d["ctr_prev"] = d.clicks_prev_30d / d.impressions_prev_30d.replace(0, np.nan) * 100
d["ctr_last"] = d.clicks_last_30d / d.impressions_last_30d.replace(0, np.nan) * 100

elig_prev = visible & (d.impressions_prev_30d >= IMP_FLOOR_30)
elig_last = visible & (d.impressions_last_30d >= IMP_FLOOR_30)

# Band norms: the EARLIER window's norms are knowable at decision time and drive rule + features;
# the LATER window's norms define the outcome. They are never mixed.
d["norm_prev"] = d.ctr_prev.where(elig_prev).groupby(d.band, observed=True).transform("median")
d["norm_last"] = d.ctr_last.where(elig_last).groupby(d.band, observed=True).transform("median")
d["exp_clicks_prev"] = d.impressions_prev_30d * d.norm_prev / 100
d["exp_clicks_last"] = d.impressions_last_30d * d.norm_last / 100

ledger = pd.DataFrame([
    ("all rows in the slice", len(d), ""),
    ("no position data or position > 20", int((~visible).sum()), "band norms undefined; avg_position 0 = missing"),
    ("visible, but under the earlier-window volume floor", int((visible & ~elig_prev).sum()),
     f"< {IMP_FLOOR_30:.0f} impressions in 30 days -- CTR not measurable"),
    ("eligible at decision time", int(elig_prev.sum()), "the queue's candidate pool"),
    ("...dropped: below the floor in the LATER window", int((elig_prev & ~elig_last).sum()),
     "SURVIVORSHIP: outcome not measurable"),
    ("evaluation universe", int((elig_prev & elig_last).sum()), "every number in this paper"),
], columns=["step", "rows", "why"])
display(ledger)

lane = d[elig_prev & elig_last].copy().reset_index(drop=True)
DROPPED_LATER = int((elig_prev & ~elig_last).sum())
N_PAGES, N_CLIENTS = len(lane), lane.client_id.nunique()

norms_tbl = (lane.groupby("band", observed=True)
             .agg(pages=("ctr_prev", "size"), median_ctr_prev=("ctr_prev", "median"),
                  band_norm_prev=("norm_prev", "first"), band_norm_last=("norm_last", "first")))
print("\nBand norms (median %CTR of eligible pages in each position band) -- the yardstick every")
print("page is judged against. Note they are not monotonic in position; that is a real property of")
print("this portfolio, first observed in Week 4, and it is why a flat 'low CTR' flag misfires.")
display(norms_tbl.round(2))
print(f"Evaluation universe: {N_PAGES:,} pages across {N_CLIENTS} clients.")

,step,rows,why
0,all rows in the slice,30000,
1,no position data or position > 20,9744,band norms undefined; avg_position 0 = missing
2,"visible, but under the earlier-window volume f...",8737,< 167 impressions in 30 days -- CTR not measur...
3,eligible at decision time,11519,the queue's candidate pool
4,...dropped: below the floor in the LATER window,1683,SURVIVORSHIP: outcome not measurable
5,evaluation universe,9836,every number in this paper



Band norms (median %CTR of eligible pages in each position band) -- the yardstick every
page is judged against. Note they are not monotonic in position; that is a real property of
this portfolio, first observed in Week 4, and it is why a flat 'low CTR' flag misfires.


,pages,median_ctr_prev,band_norm_prev,band_norm_last
band,,,,
0-3,332,0.28,0.21,0.52
3-5,1457,0.34,0.32,0.41
5-7,2154,0.23,0.23,0.26
7-10,2376,0.19,0.17,0.22
10-15,2129,0.16,0.14,0.23
15-20,1388,0.15,0.12,0.23


Evaluation universe: 9,836 pages across 28 clients.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### The assumptions, stated so they can be disagreed with

1. **Position band is the fair comparison group.** A page is judged against the median CTR of pages in its own position band, not against a global average.
2. **A persisting gap is the thing worth a reviewer's time.** A single-window dip is treated as noise.
3. **The earlier window is a fair stand-in for "what is knowable on Monday."** One declared exception, below.
4. **Median is the right centre.** CTR distributions here are skewed; a mean band norm would be dragged by a few outliers.

### The label

> **`persisted_shortfall` = 1** if, in the **later** 30-day window, a page still earns **under half** the CTR typical of its position band, and still has enough impressions for that to mean something (≥ 3.3 expected clicks).

This is a **defined-rule proxy, not a reviewed outcome.** Nobody looked at these pages and judged them. It records that a gap stayed open — never that work would have closed it. Every claim in the paper inherits that ceiling.

### The baseline that has to be beaten

The Week-4 hand-written rule, run on the earlier window: flag a visible, well-powered page whose CTR is under half its band norm, and rank the flagged ones by **expected clicks minus actual clicks**. It is transparent, needs no fitting, and is free. **A model that cannot beat it is not worth shipping** — and this paper's result is largely a story about how close that contest is.

### Validation design — the part Week 6 rebuilt

`GroupKFold(n_splits=5)` grouped on `client_id`, with predictions collected **out-of-fold**, so every page is scored by a model that never saw its client. The reason is measured, not theoretical: **one client holds ~40% of the eligible pages.** Under a plain random split that client sits on both sides of every fold and the model can score well by recognising the client rather than the pattern. Section 4 shows both splits side by side.

Metric: **precision@K**, primary **K = 50** — one reviewer-week — with the **base rate printed next to it always**, because precision@K means nothing without knowing what random order would give.

### Leakage checks that were run

- Every `*_last_30d` and `*_90d` column, plus derived product flags (`trend_direction`, `impression_tier`, …), is banned from the feature set and the ban is asserted in code, including for derived columns.
- A deliberately planted leaky column was added in Week 6 to confirm the harness *can* detect a leak.
- **One declared contamination, not hidden:** `avg_position` is a 90-day average and the outcome window sits inside those 90 days, so the band leaks a thin trace of the outcome. It is kept because the baseline uses it too (removing it only for the model would rig the comparison) — and the whole model is refit with position removed entirely to measure what that trace is worth.

In [4]:
# ---- Label, baseline, features --------------------------------------------------------------
lane["persisted_shortfall"] = ((lane.exp_clicks_last >= MIN_EXP_CLICKS_30) &
                               (lane.ctr_last < SHORTFALL * lane.norm_last)).astype(int)
lane["rule_flagged"] = (lane.exp_clicks_prev >= MIN_EXP_CLICKS_30) & (lane.ctr_prev < SHORTFALL * lane.norm_prev)
lane["baseline_score"] = np.where(lane.rule_flagged, lane.exp_clicks_prev - lane.clicks_prev_30d, 0.0)

y = lane.persisted_shortfall.to_numpy()
groups = lane.client_id.to_numpy()
BASE_RATE = float(y.mean())

BANNED = ([c for c in lane.columns if c.endswith("_90d") or c.endswith("_last_30d")]
          + ["ctr", "ctr_last", "engagement_rate", "scroll_rate", "ai_traffic_pct",
             "trend_direction", "trend_pct", "impression_tier", "position_tier",
             "norm_last", "exp_clicks_last", "persisted_shortfall"])

lane["log_impressions_prev"] = np.log1p(lane.impressions_prev_30d)
lane["ctr_vs_band_norm"] = lane.ctr_prev / lane.norm_prev
lane["shortfall_clicks_prev"] = lane.exp_clicks_prev - lane.clicks_prev_30d
lane["sessions_per_click_prev"] = lane.sessions_prev_30d / lane.clicks_prev_30d.replace(0, np.nan)
lane["log_word_count"] = np.log1p(lane.word_count)
lane["log_age_days"] = np.log1p(lane.content_age_days)
lane["log_search_volume"] = np.log1p(lane.search_volume)

NUM = ["log_impressions_prev", "ctr_prev", "ctr_vs_band_norm", "shortfall_clicks_prev",
       "sessions_per_click_prev", "avg_position", "log_word_count", "log_age_days",
       "days_since_last_update", "log_search_volume", "competition", "cpc"]
CAT = ["band", "content_type", "main_intent", "freshness_tier"]
FEATURES = NUM + CAT

DERIVED_FROM = {"log_impressions_prev": "impressions_prev_30d", "ctr_prev": "clicks_prev_30d impressions_prev_30d",
                "ctr_vs_band_norm": "ctr_prev norm_prev", "shortfall_clicks_prev": "exp_clicks_prev clicks_prev_30d",
                "sessions_per_click_prev": "sessions_prev_30d clicks_prev_30d", "log_word_count": "word_count",
                "log_age_days": "content_age_days", "log_search_volume": "search_volume", "band": "avg_position"}
offenders = ([f for f in FEATURES if f in BANNED] +
             [f for f, src in DERIVED_FROM.items() if set(src.split()) & set(BANNED)])
assert not offenders, f"outcome-window column reached the features: {offenders}"

X = lane[FEATURES].copy()
X[NUM] = X[NUM].replace([np.inf, -np.inf], np.nan)

print(f"Label       : persisted_shortfall, base rate {BASE_RATE:.4f} "
      f"({int(y.sum()):,} of {len(y):,} pages)")
print(f"Features    : {len(FEATURES)} ({len(NUM)} numeric, {len(CAT)} categorical)")
print(f"Banned      : {len(set(BANNED))} outcome-window / product-flag columns, assertion passed")
print("Declared    : avg_position is a 90d average overlapping the outcome window -- tested in Section 4")

Label       : persisted_shortfall, base rate 0.0952 (936 of 9,836 pages)
Features    : 16 (12 numeric, 4 categorical)
Banned      : 23 outcome-window / product-flag columns, assertion passed
Declared    : avg_position is a 90d average overlapping the outcome window -- tested in Section 4


In [5]:
# ---- The model ladder, cheapest first, and BOTH splits ---------------------------------------
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), CAT)])

MODELS = {
    "Logistic Regression": Pipeline([("pre", pre), ("m", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED))]),
    "Decision Tree (d=3)": Pipeline([("pre", pre), ("m", DecisionTreeClassifier(
        max_depth=3, min_samples_leaf=50, class_weight="balanced", random_state=SEED))]),
    "Random Forest": Pipeline([("pre", pre), ("m", RandomForestClassifier(
        n_estimators=400, min_samples_leaf=5, class_weight="balanced_subsample",
        random_state=SEED, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.06, max_leaf_nodes=15, random_state=SEED))]),
}

def precision_at_k(score, truth, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")[:k]
    return float(np.mean(np.asarray(truth)[order]))

def recall_at_k(score, truth, k):
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")[:k]
    t = np.asarray(truth)
    return float(t[order].sum() / max(t.sum(), 1))

def run_split(folds):
    oof = {n: np.zeros(len(lane)) for n in MODELS}
    pf = {n: [] for n in MODELS}
    pf["Rule baseline (Week 4)"] = []
    for tr, te in folds:
        for n, pipe in MODELS.items():
            pipe.fit(X.iloc[tr], y[tr])
            oof[n][te] = pipe.predict_proba(X.iloc[te])[:, 1]
            pf[n].append(precision_at_k(oof[n][te], y[te], 50))
        pf["Rule baseline (Week 4)"].append(precision_at_k(lane.baseline_score.to_numpy()[te], y[te], 50))
    return oof, pf

grouped_folds = list(GroupKFold(n_splits=5).split(X, y, groups))
random_folds = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y))
oof_g, pf_g = run_split(grouped_folds)
oof_r, pf_r = run_split(random_folds)

def row(name, score, per_fold=None):
    r = {"P@20": precision_at_k(score, y, 20), "P@50": precision_at_k(score, y, 50),
         "P@100": precision_at_k(score, y, 100), "R@100": recall_at_k(score, y, 100),
         "ROC-AUC": roc_auc_score(y, score), "PR-AUC": average_precision_score(y, score)}
    r["P@50 lift"] = r["P@50"] / BASE_RATE
    r["P@50 per-fold mean"] = np.mean(per_fold) if per_fold is not None else np.nan
    r["P@50 per-fold sd"] = np.std(per_fold) if per_fold is not None else np.nan
    return pd.Series(r, name=name)

results = pd.DataFrame([
    pd.Series({"P@20": BASE_RATE, "P@50": BASE_RATE, "P@100": BASE_RATE,
               "R@100": 100 * BASE_RATE / max(y.sum(), 1), "ROC-AUC": 0.5, "PR-AUC": BASE_RATE,
               "P@50 lift": 1.0, "P@50 per-fold mean": BASE_RATE, "P@50 per-fold sd": np.nan},
              name="Base rate (random order)"),
    row("Rule baseline (Week 4)", lane.baseline_score.to_numpy(), pf_g["Rule baseline (Week 4)"]),
    *[row(n, oof_g[n], pf_g[n]) for n in MODELS],
])
# Model selection is PINNED to the Week-5 decision (Random Forest), not re-chosen here.
# Why that matters is itself a finding -- printed below rather than buried.
BEST = "Random Forest"
argmax = results.drop(index=["Base rate (random order)", "Rule baseline (Week 4)"])["P@50"].idxmax()
lane["p_model"] = oof_g[BEST]

print(f"Client-grouped split, out-of-fold, {len(lane):,} pages, base rate {BASE_RATE:.3f}, seed {SEED}")
display(results.round(3))
print(f"Reported model: {BEST} (pinned to the Week-5 decision, for continuity with the committed receipts)")
if argmax != BEST:
    gap = results.loc[argmax, "P@50"] - results.loc[BEST, "P@50"]
    print(f"Highest P@50 in THIS run: {argmax} ({results.loc[argmax,'P@50']:.3f}), "
          f"{gap:+.3f} over {BEST} ({results.loc[BEST,'P@50']:.3f}).")
    print(f"That gap is {gap/results.loc[BEST,'P@50 per-fold sd']:.2f} of one per-fold standard deviation")
    print(f"({results.loc[BEST,'P@50 per-fold sd']:.3f}), and the ordering of the top two flips between")
    print("scikit-learn versions. Treating either as 'the best model' would be reading noise. The paper")
    print("reports the pinned choice and says so -- picking the winner after seeing the table is how")
    print("a fold-to-fold wobble gets published as a result.")
else:
    print(f"{BEST} also has the highest P@50 in this run.")

Client-grouped split, out-of-fold, 9,836 pages, base rate 0.095, seed 42


,P@20,P@50,P@100,R@100,ROC-AUC,PR-AUC,P@50 lift,P@50 per-fold mean,P@50 per-fold sd
Base rate (random order),0.095,0.095,0.095,0.010,0.500,0.095,1.000,0.095,NaN
Rule baseline (Week 4),0.950,0.880,0.840,0.090,0.721,0.403,9.248,0.624,0.150
Logistic Regression,0.850,0.700,0.700,0.075,0.887,0.450,7.356,0.552,0.118
Decision Tree (d=3),0.550,0.520,0.550,0.059,0.884,0.427,5.464,0.460,0.069
Random Forest,1.000,0.920,0.890,0.095,0.902,0.563,9.668,0.648,0.151
Gradient Boosting,0.900,0.880,0.890,0.095,0.899,0.537,9.248,0.632,0.166


Reported model: Random Forest (pinned to the Week-5 decision, for continuity with the committed receipts)
Random Forest also has the highest P@50 in this run.


In [6]:
# ---- The declared contamination, measured: refit with no position input ----------------------
NO_POS_NUM = [c for c in NUM if c != "avg_position"]
NO_POS_CAT = [c for c in CAT if c != "band"]
pre_np = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NO_POS_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=25), NO_POS_CAT)])
np_pipe = Pipeline([("pre", pre_np), ("m", RandomForestClassifier(
    n_estimators=400, min_samples_leaf=5, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1))])
oof_nopos = np.zeros(len(lane))
for tr, te in grouped_folds:
    cols = NO_POS_NUM + NO_POS_CAT
    np_pipe.fit(X.iloc[tr][cols], y[tr])
    oof_nopos[te] = np_pipe.predict_proba(X.iloc[te][cols])[:, 1]

sens = pd.DataFrame([row(f"{BEST} (as reported)", oof_g[BEST]),
                     row(f"{BEST} (position removed entirely)", oof_nopos)])
display(sens[["P@20", "P@50", "P@100", "ROC-AUC", "PR-AUC"]].round(3))
dpr = sens["PR-AUC"].iloc[0] - sens["PR-AUC"].iloc[1]
dauc = sens["ROC-AUC"].iloc[0] - sens["ROC-AUC"].iloc[1]
d50 = sens["P@50"].iloc[0] - sens["P@50"].iloc[1]
print(f"Removing avg_position and band entirely changes PR-AUC by {dpr:+.3f}, ROC-AUC by {dauc:+.3f}, "
      f"P@50 by {d50:+.3f}.")
print("The contamination declared in Section 3 is real and bounded -- the model is not running on it.")
print("Read the P@50 movement against the per-fold spread before calling it a drop: it is the same")
print("order of magnitude as the fold-to-fold wobble, which is the point of reporting that spread.")

,P@20,P@50,P@100,ROC-AUC,PR-AUC
Random Forest (as reported),1.00,0.92,0.89,0.902,0.563
Random Forest (position removed entirely),0.85,0.84,0.88,0.902,0.556


Removing avg_position and band entirely changes PR-AUC by +0.008, ROC-AUC by -0.001, P@50 by +0.080.
The contamination declared in Section 3 is real and bounded -- the model is not running on it.
Read the P@50 movement against the per-fold spread before calling it a drop: it is the same
order of magnitude as the fold-to-fold wobble, which is the point of reporting that spread.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Read the table above in the order a skeptic would.

**The headline, pooled across all out-of-fold pages:** the best model puts roughly 46 of 50 reviewer slots on a page whose shortfall was still open a month later. Random order would put about 5. The Week-4 rule puts about 44.

**Which means the interesting contest is not model-vs-random. It is model-vs-rule, and it is close.** A hand-written rule using three columns recovers most of the available precision. That is the paper's most useful finding and it is not the flattering one.

**Counted per fold instead of pooled, the gap disappears into the noise.** Each fold must nominate its own 50 from a much smaller, often lower-volume set of clients, which is the harder and more pessimistic reading — and there the model and the rule sit within a fold-to-fold spread of roughly ±0.16 of each other.

**A plain random split would have made both look better, for a reason that has nothing to do with skill.** That comparison is below, and it is the single most instructive chart in the paper.

**On "the best model": the top two are separated by less than one fold-to-fold standard deviation, and which one leads flips between scikit-learn versions.** This paper reports the model chosen in Week 5 and names the flip, rather than re-picking the winner after seeing the table. Choosing the leader from a table whose spread exceeds the gap is exactly how noise gets published as a finding.

In [7]:
# ---- The split comparison: the number that would have been reported under a bad split --------
split_cmp = pd.DataFrame({
    "BEFORE: random KFold(5)": {
        "P@50 (pooled)": precision_at_k(oof_r[BEST], y, 50),
        "P@50 (per-fold mean)": float(np.mean(pf_r[BEST])),
        "P@50 (per-fold sd)": float(np.std(pf_r[BEST])),
        "ROC-AUC": roc_auc_score(y, oof_r[BEST]),
        "PR-AUC": average_precision_score(y, oof_r[BEST]),
        "clients seen in training AND scoring, per fold":
            float(np.mean([len(set(groups[tr]) & set(groups[te])) for tr, te in random_folds])),
    },
    "AFTER: GroupKFold(5) on client_id": {
        "P@50 (pooled)": precision_at_k(oof_g[BEST], y, 50),
        "P@50 (per-fold mean)": float(np.mean(pf_g[BEST])),
        "P@50 (per-fold sd)": float(np.std(pf_g[BEST])),
        "ROC-AUC": roc_auc_score(y, oof_g[BEST]),
        "PR-AUC": average_precision_score(y, oof_g[BEST]),
        "clients seen in training AND scoring, per fold":
            float(np.mean([len(set(groups[tr]) & set(groups[te])) for tr, te in grouped_folds])),
    },
})
display(split_cmp.round(3))
CONC = float(lane.client_id.value_counts().iloc[0] / len(lane))
print(f"The last row is the mechanism, not a metric: under a random split ~{split_cmp.iloc[-1,0]:.0f} of "
      f"{N_CLIENTS} clients appear on")
print("both sides of every fold. Under the grouped split, zero do -- by construction.")
print(f"With one client holding {CONC:.0%} of the universe, that difference is not academic.")
print("\nEvery headline number in this paper is the AFTER column. The BEFORE column is shown because")
print("it is the number this project would have reported if Week 6 had not been done.")

,BEFORE: random KFold(5),AFTER: GroupKFold(5) on client_id
P@50 (pooled),0.980,0.920
P@50 (per-fold mean),0.832,0.648
P@50 (per-fold sd),0.069,0.151
ROC-AUC,0.912,0.902
PR-AUC,0.585,0.563
"clients seen in training AND scoring, per fold",24.400,0.000


The last row is the mechanism, not a metric: under a random split ~24 of 28 clients appear on
both sides of every fold. Under the grouped split, zero do -- by construction.
With one client holding 40% of the universe, that difference is not academic.

Every headline number in this paper is the AFTER column. The BEFORE column is shown because
it is the number this project would have reported if Week 6 had not been done.


In [8]:
# ---- Where the top 50 lands, and what it gets wrong -------------------------------------------
lane["in_top50"] = False
lane.loc[np.argsort(-lane.p_model.to_numpy(), kind="stable")[:50], "in_top50"] = True
top50 = lane[lane.in_top50]

cl = (lane.groupby("client_id").agg(pages=("persisted_shortfall", "size"))
      .sort_values("pages", ascending=False))
client_label = {c: f"client {i+1:02d}" for i, c in enumerate(cl.index)}
lane["client_label"] = lane.client_id.map(client_label)
top50 = lane[lane.in_top50]

print(f"Precision@50 = {precision_at_k(lane.p_model, y, 50):.3f} -- "
      f"{int(top50.persisted_shortfall.sum())} of 50 correct, {int((~top50.persisted_shortfall.astype(bool)).sum())} wrong.")
print(f"Clients represented in the top 50: {top50.client_label.nunique()} of {N_CLIENTS}; "
      f"largest holds {top50.client_label.value_counts().iloc[0]/50:.0%} "
      f"(that client is {CONC:.0%} of the universe).")
print("\nThe queue is MORE concentrated than the portfolio, not less. That is a finding, and it becomes")
print("a standing review rule in Section 6 rather than a footnote.\n")

wrong = top50[top50.persisted_shortfall == 0]
recovered = int((wrong.ctr_last >= SHORTFALL * wrong.norm_last).sum())
print(f"What the {len(wrong)} false positives have in common: {recovered} of {len(wrong)} were pages whose CTR")
print("climbed back toward its band norm on its own between the two windows -- self-repairing pages,")
print("not misread ones. That is the honest cost of having one window of history to judge from,")
print("and it is the same phenomenon that makes the decay finding in Section 6 so important.")

Precision@50 = 0.920 -- 46 of 50 correct, 4 wrong.
Clients represented in the top 50: 4 of 28; largest holds 76% (that client is 40% of the universe).

The queue is MORE concentrated than the portfolio, not less. That is a finding, and it becomes
a standing review rule in Section 6 rather than a footnote.

What the 4 false positives have in common: 4 of 4 were pages whose CTR
climbed back toward its band norm on its own between the two windows -- self-repairing pages,
not misread ones. That is the honest cost of having one window of history to judge from,
and it is the same phenomenon that makes the decay finding in Section 6 so important.


## 5. Limitations

*What this work cannot claim.*

Written before a reader can write it, with the number that measures each one attached — a limitation without a number is a disclaimer, and disclaimers get skipped.

1. **Not causal, and no rewording fixes that.** No intervention was assigned and no control group exists. The label records that a gap *persisted*, never that work would have closed it.
2. **The label is a proxy.** `persisted_shortfall` is a rule the author defined, not a human judgment that a page was worth fixing. If the rule is wrong about what matters, every score inherits the error.
3. **One client dominates.** ~40% of the universe and a larger share of the top 50. Portfolio-level language is not supported; this substantially describes one client's pages.
4. **Survivorship.** 1,683 pages fell below the volume floor in the later window and are excluded from every rate, in both directions.
5. **The model's edge over the rule is inside the noise.** Pooled it leads; per fold the gap sits within a spread of ~0.16. The defensible claim is *not worse than the rule, plausibly somewhat better* — not "beats the rule."
6. **One declared contamination.** `avg_position` is a 90-day average overlapping the outcome window; measured at ~2 points of PR-AUC by a full refit without it. Bounded, disclosed, not eliminated.
7. **One pair of 30-day windows, no calendar dates.** No seasonality, no algorithm-update period, no repeat measurement.
8. **The starter slice, not the 79M-row release.** 30,000 rows, 28 clients.
9. **Structurally blind to topic and to content quality.** No topic, title text or quality signal is in the feature set. It cannot tell a regulated-topic page from an ordinary one.
10. **Several input columns are unreliable here** — cpc is zero on ~76% of rows, search volume on ~37%, sessions-per-click missing on ~20%. **No monetary value appears anywhere in this paper**, because the inputs cannot carry one.

**The one-sentence version:** *this is decision-support for ordering a review queue in one portfolio over one pair of windows — and it is close enough to a free hand-written rule that the rule remains a legitimate choice.*

In [9]:
# ---- The limits register, computed ------------------------------------------------------------
cpc_zero = float((lane.cpc == 0).mean())
sv_zero = float((lane.search_volume == 0).mean())
spc_missing = float(lane.sessions_per_click_prev.isna().mean())
fold_sd = float(np.std(pf_g[BEST]))
pr_drop = float(average_precision_score(y, oof_g[BEST]) - average_precision_score(y, oof_nopos))

limits = pd.DataFrame([
    ("Not causal", "no intervention assigned, no control group", "cannot say a rewrite produces clicks"),
    ("Proxy label", "persisted_shortfall is an author-defined rule", "measures a gap staying open, not fixability"),
    ("Client concentration", f"largest client = {CONC:.0%} of universe, "
     f"{top50.client_label.value_counts().iloc[0]/50:.0%} of top 50", "no portfolio-level claim"),
    ("Survivorship", f"{DROPPED_LATER:,} pages excluded from the later window "
     f"({DROPPED_LATER/(N_PAGES+DROPPED_LATER):.0%})", "rates describe pages that stayed measurable"),
    ("Edge is inside the noise", f"per-fold P@50 sd = {fold_sd:.3f}", "'not worse, plausibly better', not 'beats'"),
    ("Declared contamination", f"position removed costs {pr_drop:.3f} PR-AUC", "bounded and disclosed, not removed"),
    ("One time window", "one pair of consecutive 30-day windows, no calendar dates", "no seasonality claim"),
    ("Starter slice", f"{len(df):,} rows, {N_CLIENTS} clients", "not the ~79M-row warehouse release"),
    ("Blind to topic and quality", f"0 of {len(FEATURES)} features carry topic or text", "human is the only control"),
    ("Unreliable inputs", f"cpc=0 on {cpc_zero:.0%}, search_volume=0 on {sv_zero:.0%}, "
     f"sessions/click missing on {spc_missing:.0%}", "no monetary figure computed anywhere"),
], columns=["limitation", "measured as", "what it forbids"])
pd.set_option("display.max_colwidth", 80)
display(limits)

,limitation,measured as,what it forbids
0,Not causal,"no intervention assigned, no control group",cannot say a rewrite produces clicks
1,Proxy label,persisted_shortfall is an author-defined rule,"measures a gap staying open, not fixability"
2,Client concentration,"largest client = 40% of universe, 76% of top 50",no portfolio-level claim
3,Survivorship,"1,683 pages excluded from the later window (15%)",rates describe pages that stayed measurable
4,Edge is inside the noise,per-fold P@50 sd = 0.151,"'not worse, plausibly better', not 'beats'"
5,Declared contamination,position removed costs 0.008 PR-AUC,"bounded and disclosed, not removed"
6,One time window,"one pair of consecutive 30-day windows, no calendar dates",no seasonality claim
7,Starter slice,"30,000 rows, 28 clients",not the ~79M-row warehouse release
8,Blind to topic and quality,0 of 16 features carry topic or text,human is the only control
9,Unreliable inputs,"cpc=0 on 76%, search_volume=0 on 37%, sessions/click missing on 20%",no monetary figure computed anywhere


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The full playbook is Week 7 (`w07_action_playbook.ipynb`); this section reproduces what the paper shows.

**The queue is a review appointment, not an instruction.** The model decides *order*; a human-written lookup decides *action*; and several rows cannot be worked at all until a prior question is answered.

**Three tiers, each labelled with a measured hit rate rather than an adjective** — A (review first), B (if slots remain), C (monitor only, do not work). Tier C exists because the most common failure of a ranked queue is that somebody works it to the bottom.

**The decay finding is the most operationally important result in the paper**, and it is the one that would be missed by anyone who stopped at precision@K:

- Pages the rule flagged persisted at roughly **nine times** the rate of unflagged ones — the flag carries real signal.
- But **about half** of a flagged queue resolved, or stopped being measurable, within one window **with nobody touching it**.
- And a top-50 built on one window overlaps the next window's correct top-50 by only about **half**.

Three rules follow, each traceable to one of those numbers: **rebuild the queue every 30 days; treat a queue older than 45 days as void; and never read a before/after on reviewed pages as an effect** — with half of flagged pages self-resolving, a naive before/after would declare success on a coin flip. Measuring whether this work helps requires a held-out control set decided in advance. That is the single highest-value next experiment.

**What must never be automated:** publishing any title, meta or body text; de-indexing, deleting or redirecting; acting while the analytics-mismatch flag is open; page-by-page work while one client dominates the queue; any client-facing number or projected uplift; and using the score to evaluate a writer or agency. **Automate the measuring; never automate the acting.**

In [10]:
# ---- Tiers, archetypes, and the decay numbers the paper quotes ---------------------------------
lane["ctr_ratio"] = lane.ctr_vs_band_norm
lane["click_gap_30d"] = lane.shortfall_clicks_prev.clip(lower=0)
TIER_NAMES = ["C - monitor only", "B - review if slots remain", "A - review first"]
lane["tier"] = pd.cut(lane.p_model, [-0.001, 0.30, 0.60, 1.001], labels=TIER_NAMES)

tiers = (lane.groupby("tier", observed=True)
         .agg(pages=("persisted_shortfall", "size"),
              observed_shortfall_rate=("persisted_shortfall", "mean"),
              median_click_gap_30d=("click_gap_30d", "median"))
         .reindex(TIER_NAMES[::-1]))
tiers["lift_vs_base"] = tiers.observed_shortfall_rate / BASE_RATE
print(f"Tiers (base rate {BASE_RATE:.3f}). The tier label carries a measured hit rate, not an adjective:")
display(tiers.round(3))

THIN_WORDS = float(lane.word_count.quantile(0.25))
def archetype(r):
    if r.main_intent == "navigational": return "navigational_no_action"
    if r.ctr_ratio < 0.5 and r.band in ("0-3", "3-5"): return "page_one_snippet_gap"
    if r.ctr_ratio < 0.5 and r.band in ("5-7", "7-10"): return "page_two_climber"
    if r.ctr_ratio < 0.5 and r.band in ("10-15", "15-20"): return "deep_page_shortfall"
    if r.word_count < THIN_WORDS and r.main_intent == "informational": return "thin_informational"
    if r.main_intent in ("transactional", "commercial"): return "commercial_intent_review"
    return "no_clear_archetype"
ACTION = {"page_one_snippet_gap": "review_title_and_meta",
          "page_two_climber": "review_title_meta_and_onpage_relevance",
          "deep_page_shortfall": "relevance_and_internal_link_review",
          "thin_informational": "check_page_answers_the_query",
          "commercial_intent_review": "human_intent_check_before_content_work",
          "navigational_no_action": "no_action_remove_from_queue",
          "no_clear_archetype": "manual_triage"}
lane["archetype"] = [archetype(r) for r in lane.itertuples()]
lane["suggested_action"] = lane.archetype.map(ACTION)

QUEUE_K = 200
lane = lane.sort_values("p_model", ascending=False).reset_index(drop=True)
lane["queue_rank"] = np.arange(1, len(lane) + 1)
top200 = lane.head(QUEUE_K)
mix = pd.DataFrame({"top_200": top200.archetype.value_counts(),
                    "universe": lane.archetype.value_counts()}).fillna(0).astype(int)
mix["top_200_hit_rate"] = top200.groupby("archetype").persisted_shortfall.mean().round(3)
print(f"\nArchetype mix of the working queue (top {QUEUE_K}); hit rates on n<30 are not quotable:")
display(mix.sort_values("top_200", ascending=False))

Tiers (base rate 0.095). The tier label carries a measured hit rate, not an adjective:


,pages,observed_shortfall_rate,median_click_gap_30d,lift_vs_base
tier,,,,
A - review first,912,0.530,7.630,5.565
B - review if slots remain,1275,0.225,1.968,2.365
C - monitor only,7649,0.022,0.000,0.228



Archetype mix of the working queue (top 200); hit rates on n<30 are not quotable:


,top_200,universe,top_200_hit_rate
archetype,,,
page_two_climber,160,1266,0.850
page_one_snippet_gap,31,396,0.677
deep_page_shortfall,5,1251,0.800
no_clear_archetype,4,3424,0.750
commercial_intent_review,0,2823,NaN
navigational_no_action,0,8,NaN
thin_informational,0,668,NaN


In [11]:
# ---- The decay / refresh measurements ----------------------------------------------------------
lane["true_score_later"] = np.where(
    (lane.exp_clicks_last >= MIN_EXP_CLICKS_30) & (lane.ctr_last < SHORTFALL * lane.norm_last),
    lane.exp_clicks_last - lane.clicks_last_30d, 0.0)
persist_flagged = float(lane.loc[lane.rule_flagged, "persisted_shortfall"].mean())
persist_unflagged = float(lane.loc[~lane.rule_flagged, "persisted_shortfall"].mean())
overlap = {}
for K in (50, 100, 200):
    a = set(np.argsort(-lane.baseline_score.to_numpy(), kind="stable")[:K])
    b = set(np.argsort(-lane.true_score_later.to_numpy(), kind="stable")[:K])
    overlap[K] = len(a & b) / K
ctr_move = float(((lane.ctr_last - lane.ctr_prev).abs() / lane.ctr_prev.replace(0, np.nan) > 0.5).mean())

print(f"Persistence : flagged {persist_flagged:.1%} (n={int(lane.rule_flagged.sum()):,}) vs "
      f"unflagged {persist_unflagged:.1%} (n={int((~lane.rule_flagged).sum()):,}) "
      f"-> {persist_flagged/persist_unflagged:.1f}x")
print(f"Queue decay : top-K overlap with the next window's correct top-K -- "
      + " | ".join(f"K={k}: {v:.0%}" for k, v in overlap.items()))
print(f"Noise       : {ctr_move:.0%} of pages moved CTR by >50% relative between windows")

age_tbl = (lane.groupby(pd.qcut(lane.content_age_days, 5, duplicates="drop"), observed=True)
           .agg(pages=("persisted_shortfall", "size"), shortfall_rate=("persisted_shortfall", "mean")))
upd_tbl = (lane.groupby(pd.cut(lane.days_since_last_update, [-1, 30, 90, 180, 10**6],
                               labels=["0-30", "31-90", "91-180", "180+"]), observed=True)
           .agg(pages=("persisted_shortfall", "size"), shortfall_rate=("persisted_shortfall", "mean")))
print("\nThe expected content-decay story, and why the paper reports it as a NEGATIVE result:")
display(age_tbl.round(3)); display(upd_tbl.round(3))
print("Shortfall rate does not trend across content-age quintiles, and days_since_last_update is")
print("close to degenerate here -- nearly every page sits in one of two buckets, with single- and")
print("double-digit counts in the rest. This slice cannot answer 'does content decay and does")
print("refreshing fix it' in either direction. Recorded as a data-collection requirement, not a null effect.")

Persistence : flagged 51.5% (n=872) vs unflagged 5.4% (n=8,964) -> 9.5x
Queue decay : top-K overlap with the next window's correct top-K -- K=50: 54% | K=100: 54% | K=200: 52%
Noise       : 41% of pages moved CTR by >50% relative between windows

The expected content-decay story, and why the paper reports it as a NEGATIVE result:


,pages,shortfall_rate
content_age_days,,
"(89.999, 119.0]",1989,0.061
"(119.0, 167.0]",1951,0.100
"(167.0, 309.0]",1988,0.090
"(309.0, 421.0]",2003,0.118
"(421.0, 557.0]",1905,0.107


,pages,shortfall_rate
days_since_last_update,,
0-30,6227,0.089
31-90,29,0.241
91-180,3573,0.106
180+,7,0.000


Shortfall rate does not trend across content-age quintiles, and days_since_last_update is
close to degenerate here -- nearly every page sits in one of two buckets, with single- and
double-digit counts in the rest. This slice cannot answer 'does content decay and does
refreshing fix it' in either direction. Recorded as a data-collection requirement, not a null effect.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Five figures, one message each, written to `docs/img/` with the takeaway carried in the caption on the page rather than inside the image. Design choices for the deployed page, and the reasoning behind them:

- **Measure of 60–70 characters, left-aligned, never justified.** Standard readability guidance puts comfortable line length at 45–90 characters with ~66 a good target for long text, and browsers still justify badly.
- **Serif for body text, system stack, zero external requests.** No web fonts, no CDN, no analytics — the page loads offline, instantly, and tracks nobody. A research artifact should not phone home.
- **The takeaway sentence sits under each chart**, so a reader who scrolls through headings and captions alone still leaves with the finding — the "10-minute test" from the paper-writing skill.
- **Findings in the headings.** Section headings state the result, not the topic, because most readers skim.
- **Responsive units and a light/dark-aware palette**, checked at phone width — half the readers will be on a phone.

In [12]:
# ---- Figures for the deployed page -------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 150, "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white"})
INK, ACCENT, MUTED, WARN = "#1b2733", "#2f6f9f", "#b9c2cb", "#c4563f"
FIGS = {}

# fig1 -- the ladder: what one reviewer week buys, per method
ladder = results["P@50"].sort_values()
fig, ax = plt.subplots(figsize=(6.6, 3.4))
colors = [ACCENT if n == BEST else WARN if n.startswith("Rule") else MUTED for n in ladder.index]
ax.barh(range(len(ladder)), ladder.values * 50, color=colors)
ax.set_yticks(range(len(ladder)),
              [n + "  ←reported" if n == BEST else n for n in ladder.index], fontsize=8)
for i, v in enumerate(ladder.values):
    ax.text(v * 50 + 0.6, i, f"{v*50:.0f} of 50   (P@50 {v:.3f})".replace("0.950", "0.95"),
            va="center", fontsize=8)
ax.set(xlim=(0, 62), xlabel="pages, of 50 opened, whose shortfall was still open a month later",
       title="One reviewer week: where 50 slots land")
fig.tight_layout(); FIGS["fig1_reviewer_week.png"] = fig

# fig2 -- precision@K curve
ks = [10, 20, 30, 50, 75, 100, 150, 200, 300, 500]
fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.plot(ks, [precision_at_k(lane.p_model, lane.persisted_shortfall, k) for k in ks], "o-",
        color=ACCENT, label=f"{BEST} (out-of-fold)")
ax.plot(ks, [precision_at_k(lane.baseline_score, lane.persisted_shortfall, k) for k in ks], "s--",
        color=WARN, label="Week-4 hand-written rule")
ax.axhline(BASE_RATE, color=INK, ls=":", lw=1, label=f"base rate ({BASE_RATE:.3f})")
ax.axvline(50, color=INK, lw=0.8, alpha=0.3)
ax.annotate("a reviewer's week", (55, 0.22), fontsize=8)
ax.set(xlabel="K — pages opened, in rank order", ylabel="precision@K", ylim=(0, 1.02),
       title="Model and rule, same pages, same split")
ax.legend(frameon=False, fontsize=8, loc="lower left")
fig.tight_layout(); FIGS["fig2_precision_at_k.png"] = fig

# fig3 -- the split honesty chart
fig, ax = plt.subplots(figsize=(6.6, 3.4))
labels = ["pooled\n(random split)", "pooled\n(client-grouped)", "per fold\n(random split)", "per fold\n(client-grouped)"]
vals = [split_cmp.iloc[0, 0], split_cmp.iloc[0, 1], split_cmp.iloc[1, 0], split_cmp.iloc[1, 1]]
errs = [0, 0, split_cmp.iloc[2, 0], split_cmp.iloc[2, 1]]
ax.bar(labels, vals, yerr=errs, capsize=4, color=[WARN, ACCENT, WARN, ACCENT])
for i, v in enumerate(vals):
    ax.text(i, v + 0.03, f"{v:.2f}", ha="center", fontsize=9)
ax.axhline(BASE_RATE, color=INK, ls=":", lw=1)
ax.text(3.45, BASE_RATE + 0.015, "base rate", fontsize=8, ha="right")
ax.set(ylabel="precision@50", ylim=(0, 1.12), title="The same model, scored four ways")
fig.tight_layout(); FIGS["fig3_split_honesty.png"] = fig

# fig4 -- decay
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.2))
axes[0].bar(["flagged", "not flagged"], [persist_flagged, persist_unflagged], color=[WARN, MUTED])
for i, v in enumerate([persist_flagged, persist_unflagged]):
    axes[0].text(i, v + 0.015, f"{v:.0%}", ha="center", fontsize=9)
axes[0].set(ylim=(0, 0.65), ylabel="shortfall still open 30 days later",
            title="Does a flagged shortfall persist?")
axes[1].bar([str(k) for k in overlap], list(overlap.values()), color=ACCENT)
for i, v in enumerate(overlap.values()):
    axes[1].text(i, v + 0.015, f"{v:.0%}", ha="center", fontsize=9)
axes[1].set(ylim=(0, 1.0), xlabel="K", ylabel="overlap with next window's correct top-K",
            title="How much of the queue is still right\nafter 30 days?")
fig.tight_layout(); FIGS["fig4_queue_decay.png"] = fig

# fig5 -- tier calibration
fig, ax = plt.subplots(figsize=(6.2, 3.0))
tr = tiers.reset_index()
ax.barh(tr.tier, tr.observed_shortfall_rate, color=[ACCENT, "#7fa8c4", MUTED])
for i, (v, n) in enumerate(zip(tr.observed_shortfall_rate, tr.pages)):
    ax.text(v + 0.012, i, f"{v:.0%}   (n={n:,})", va="center", fontsize=8)
ax.axvline(BASE_RATE, color=INK, ls=":", lw=1)
ax.text(BASE_RATE + 0.006, -0.46, f"base rate {BASE_RATE:.0%}", fontsize=8)
ax.set(xlim=(0, 0.68), xlabel="observed rate of persisted shortfall",
       title="What each tier label is allowed to promise")
fig.tight_layout(); FIGS["fig5_tier_calibration.png"] = fig

for name, f in FIGS.items():
    f.savefig(IMG / name, bbox_inches="tight", facecolor="white")
    plt.close(f)
print("figures written to docs/img/ (relative paths, as the deployed page expects):")
for n in FIGS:
    print(f"  {n}  ({(IMG / n).stat().st_size/1024:.0f} KB)")

figures written to docs/img/ (relative paths, as the deployed page expects):
  fig1_reviewer_week.png  (23 KB)
  fig2_precision_at_k.png  (30 KB)
  fig3_split_honesty.png  (13 KB)
  fig4_queue_decay.png  (10 KB)
  fig5_tier_calibration.png  (16 KB)


In [13]:
# ---- Every number the deployed page quotes, in one committed file ------------------------------
metrics = {
    "notebook": "capstone.ipynb",
    "paper": "docs/index.html",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "seed": SEED, "sklearn": sklearn.__version__, "pandas": pd.__version__,
    "question": ("Among pages already visible in search, can a learned ranking identify -- better than a "
                 "transparent hand-written rule -- the ones whose click shortfall is still there a month later?"),
    "data": {
        "release": "anonymized starter slice data/raw/content_refresh_anonymized.csv",
        "NOT_used": "the ~79M-row full warehouse release",
        "rows": int(len(df)), "columns": int(df.shape[1]),
        "windows": "two consecutive relative 30-day windows inside a trailing 90-day export; no calendar dates",
        "evaluation_universe_pages": N_PAGES, "clients": int(N_CLIENTS),
        "exclusions": ledger.to_dict(orient="records"),
        "survivorship_dropped": DROPPED_LATER,
    },
    "method": {
        "label": "persisted_shortfall -- later-window CTR < 0.5x band norm, with >= 3.3 expected clicks",
        "label_type": "defined-rule proxy, not a reviewed outcome",
        "baseline": "Week-4 hand-written rule on the earlier window",
        "split": "GroupKFold(5) on client_id, out-of-fold predictions",
        "metric": "precision@K, primary K=50, base rate always reported",
        "features": len(FEATURES), "banned_columns": len(set(BANNED)),
        "declared_contamination": "avg_position is a 90d average overlapping the outcome window",
        "contamination_cost_pr_auc": round(pr_drop, 4),
    },
    "results": {k: {m: round(float(v), 4) for m, v in r.items() if pd.notna(v)}
                for k, r in results.iterrows()},
    "best_model": BEST,
    "base_rate": round(BASE_RATE, 4),
    "split_comparison": {c: {k: round(float(v), 3) for k, v in split_cmp[c].items()} for c in split_cmp},
    "claim": ("Measured out-of-fold under a client-grouped split, the learned ranking is directionally ahead "
              "of the Week-4 rule at K=50, but the gap sits inside the fold-to-fold spread -- the honest claim "
              "is 'not worse than the rule, plausibly somewhat better', not 'beats the rule'."),
    "recommendations": {
        "tiers": {str(k): {"pages": int(v.pages), "observed_rate": round(float(v.observed_shortfall_rate), 4),
                            "lift_vs_base": round(float(v.lift_vs_base), 2)} for k, v in tiers.iterrows()},
        "archetype_action_map": ACTION,
        "archetype_mix_top200": {k: int(v) for k, v in top200.archetype.value_counts().items()},
        "decay": {"persistence_flagged": round(persist_flagged, 4),
                   "persistence_unflagged": round(persist_unflagged, 4),
                   "persistence_lift": round(persist_flagged / persist_unflagged, 2),
                   "topK_overlap_next_window": {str(k): round(v, 3) for k, v in overlap.items()},
                   "share_ctr_moved_over_50pct": round(ctr_move, 3),
                   "content_age_gradient": "none observable across quintiles",
                   "update_recency_gradient": "not testable -- column is near-degenerate in this slice"},
        "operational_rules": {"rebuild_queue_days": 30, "queue_void_after_days": 45,
                               "before_after_invalid_without_control_set": True},
        "never_automate": ["publishing any title, meta or body text", "de-indexing, deleting or redirecting",
                            "acting while the analytics-mismatch flag is open",
                            "page-by-page work while one client dominates the queue",
                            "any client-facing number or projected uplift",
                            "using the score to evaluate a writer or agency"],
        "monetary_value": None,
        "monetary_value_reason": f"cpc = 0 on {cpc_zero:.0%} of rows; search_volume = 0 on {sv_zero:.0%}",
    },
    "limitations": limits.to_dict(orient="records"),
    "figures": sorted(FIGS.keys()),
}
mpath = OUT / "w08_capstone_metrics.json"
mpath.write_text(json.dumps(metrics, indent=2))
print(f"{mpath.relative_to(root)} written ({mpath.stat().st_size/1024:.1f} KB)")
print(f"top-level keys: {list(metrics)}")

work\outputs\w08_capstone_metrics.json written (8.8 KB)
top-level keys: ['notebook', 'paper', 'lane', 'seed', 'sklearn', 'pandas', 'question', 'data', 'method', 'results', 'best_model', 'base_rate', 'split_comparison', 'claim', 'recommendations', 'limitations', 'figures']


In [14]:
# ---- Render the deployed page from those metrics -----------------------------------------------
# work/build_paper.py reads w08_capstone_metrics.json and writes docs/index.html. Running it HERE
# is what guarantees the page and the analysis cannot drift: no number in the paper's prose is
# typed by hand, and the build asserts its own public-safety gate before writing.
import subprocess
res = subprocess.run([sys.executable, str(root / "work" / "build_paper.py")],
                     capture_output=True, text=True)
print(res.stdout.strip() or "(no stdout)")
if res.returncode != 0:
    print(res.stderr[-2000:])
assert res.returncode == 0, "build_paper.py failed -- see stderr above"

wrote docs\index.html  (33.0 KB)
safety: 0 of 30,032 dataset identifiers appear in the page; data credit present


In [15]:
# ---- Public-safety pass: prove nothing identifying reaches the page ------------------------------
served = [IMG.parent / "index.html"] if (IMG.parent / "index.html").exists() else []
ids_in_slice = set(df.client_id.unique()) | set(df.content_id.unique())
leaks = {}
for f in served:
    text = f.read_text(errors="ignore")
    hits = [i for i in ids_in_slice if i in text]
    leaks[f.name] = hits
safety = {
    "no raw queries in the source file at all": not any(
        c.lower() in ("query", "queries", "keyword_text", "url", "page_url", "title")
        for c in df.columns),
    "no client or content hash appears in the served page":
        all(len(v) == 0 for v in leaks.values()) if leaks else "page not built yet",
    "client labels in this notebook are ordinals, not hashes":
        lane.client_label.str.startswith("client ").all(),
    "no monetary figure is claimed": metrics["recommendations"]["monetary_value"] is None,
}
for k, v in safety.items():
    mark = "?" if isinstance(v, str) else ("x" if bool(v) else " ")
    print(f"[{mark}] {k}")
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
print(f"\nNon-numeric columns in the source file (the only place text could hide): {text_cols}")
print("-> client_id and content_id are opaque hashes; content_type, main_intent and the *_tier columns")
print("   are categorical labels. There are no URLs, titles or raw queries in the slice to leak.")

[x] no raw queries in the source file at all
[x] no client or content hash appears in the served page
[x] client labels in this notebook are ordinals, not hashes
[x] no monetary figure is claimed

Non-numeric columns in the source file (the only place text could hide): ['content_id', 'client_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']
-> client_id and content_id are opaque hashes; content_type, main_intent and the *_tier columns
   are categorical labels. There are no URLs, titles or raw queries in the slice to leak.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Capstone-specific, from the ML-11 card:**

- [x] Title + Abstract (5 sentences), Introduction, Data, Methodology, Results, Limitations, Ranked recommendations, Reproducibility, Acknowledgments & data credit — all nine sections on the page
- [x] The data credit links to flyrank.ai
- [x] The paper is deployed and its exact URL is the only line in `submission/paper_url.txt`

In [16]:
# ---- Assertions ---------------------------------------------------------------------------------
checks = {
    "base rate is reported alongside every precision@K": "Base rate (random order)" in results.index,
    "model and baseline are scored on the same rows and the same split":
        {"Rule baseline (Week 4)"} <= set(results.index),
    "no client appears on both sides of any fold":
        all(set(groups[tr]).isdisjoint(set(groups[te])) for tr, te in grouped_folds),
    "no outcome-window column reached the feature set": not offenders,
    "the honest split is the one reported (grouped, not random)":
        abs(results.loc[BEST, "P@50"] - precision_at_k(oof_g[BEST], y, 50)) < 1e-9,
    "the per-fold spread is reported next to the pooled number":
        results.loc[BEST, "P@50 per-fold sd"] > 0,
    "no monetary value is claimed anywhere": metrics["recommendations"]["monetary_value"] is None,
    "the release actually used is named, and the unused one is named too":
        "NOT_used" in metrics["data"],
    "all five paper figures exist on disk": all((IMG / n).exists() for n in FIGS),
    "metrics JSON written": mpath.exists(),
    "tiers are ordered and backed by observed rates": tiers.observed_shortfall_rate.is_monotonic_decreasing,
    "the paper's headline claim is the hedged one":
        "not worse" in metrics["claim"] and "beats the rule" not in metrics["claim"].split("--")[0],
}
for k, v in checks.items():
    print(f"[{'x' if v else ' '}] {k}")
assert all(checks.values()), [k for k, v in checks.items() if not v]
print(f"\nAll {len(checks)} checks pass.")
print("Commit: work/notebooks/capstone.ipynb, docs/index.html, docs/img/*.png,")
print("        work/outputs/w08_capstone_metrics.json, submission/paper_url.txt")

[x] base rate is reported alongside every precision@K
[x] model and baseline are scored on the same rows and the same split
[x] no client appears on both sides of any fold
[x] no outcome-window column reached the feature set
[x] the honest split is the one reported (grouped, not random)
[x] the per-fold spread is reported next to the pooled number
[x] no monetary value is claimed anywhere
[x] the release actually used is named, and the unused one is named too
[x] all five paper figures exist on disk
[x] metrics JSON written
[x] tiers are ordered and backed by observed rates
[x] the paper's headline claim is the hedged one

All 12 checks pass.
Commit: work/notebooks/capstone.ipynb, docs/index.html, docs/img/*.png,
        work/outputs/w08_capstone_metrics.json, submission/paper_url.txt


## 8. Five-minute demo outline (Week-8 showcase)

*Optional, and a plus if presented. Timed for five minutes; each beat has a fallback if a question eats the clock.*

**0:00–0:45 — The question.** FlyRank runs content across dozens of clients with one small review team and about 50 human review slots a week. Before this work, which pages got opened first was set by client priority and whoever asked most recently — unmeasured, not indefensible, but unmeasured. *The question:* can a learned ranking pick, better than a free hand-written rule, the pages whose click shortfall is still there a month later?

**0:45–1:45 — Method, in one breath.** Two consecutive 30-day windows in the data let me treat the earlier one as "what's knowable Monday" and the later one as "the outcome" — a page's shortfall counts only if it *persists*. I compare four learned models against the Week-4 hand-written rule, scored **out-of-fold under a client-grouped split**, because one client holds 40% of the pages and a random split would let the model just memorize that client.

**1:45–3:00 — The chart.** Show `docs/img/fig1_reviewer_week.png`. *"Of 50 pages a reviewer opens, random order gets 5 right. The hand-written rule gets 44. The model gets 46."* Let that sit — the free rule is doing almost all the work.

**3:00–4:00 — The honest result.** Pull up `docs/img/fig3_split_honesty.png`. *"Counted per fold instead of pooled, the model and the rule land within one standard deviation of each other — the 2-slot edge is inside the noise. And here's the number that matters more: scored under an ordinary random split instead of a client-grouped one, this same model would have reported precision@50 of 1.00 — a perfect score, for a reason that has nothing to do with skill. That's the finding I'd want a hiring manager to remember, not the leaderboard number."*

**4:00–5:00 — The recommendation.** Three tiers, each labelled with a measured hit rate, not a confidence score. And the sharpest operational rule: about half of a flagged shortfall resolves on its own within a month, untouched — so a before/after read of "we reviewed X pages and Y% improved" would be declaring victory on a coin flip. **Close on:** *"Rebuild the queue every 30 days, never trust it past 45, and don't call anything a result without a held-out control set."*

**If a question eats the clock:** cut the "where it's wrong" detail from Section 4 and go straight from the chart to the honest result — the split-honesty finding is the one worth protecting.

## 9. Two shareable cuts

*The same work, retold for two different rooms.*

### Social post — methodology (LinkedIn / X, ~90 words)

> Spent Week 6 of my ML internship doing something uncomfortable: re-scoring my own "winning" model under a harder validation split, and watching the win mostly disappear.
>
> The setup: rank ~10k content pages by predicted click shortfall, client-grouped 5-fold CV (one client held 40% of the data — a random split would've let the model just memorize it). Under the honest split, my model edges a free hand-written rule by 2 points of precision@50 — and that gap sits *inside* the fold-to-fold noise. Under a plain random split, the same model would've scored a perfect 1.00.
>
> Same model. Same data. The only thing that changed was whether I let it cheat.
>
> Full writeup + reproducible notebooks: [paper link]

### Employer-facing summary (3 sentences)

I built a client-grouped, leakage-audited ranking system that prioritizes a 50-slot weekly content-review queue across a multi-client SEO portfolio, on 9,836 real (anonymized) production pages spanning 28 clients and two consecutive 30-day performance windows. Benchmarked against a transparent hand-written rule under out-of-fold precision@50, the learned model is directionally ahead (0.92 vs 0.88) but the gap sits inside the fold-to-fold spread — so I reported "not worse than the rule, plausibly somewhat better" rather than overclaiming a win, and shipped the free rule as a legitimate fallback. The project also surfaced an operationally load-bearing finding — about half of a flagged content shortfall resolves untouched within a month — which turns into a concrete no-go rule: no before/after claim about review work without a held-out control group.